<a href="https://colab.research.google.com/github/AleenaNizar-Analyst/UAE-Logistics-Intelligence-Platform/blob/main/UAE_Logistics_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import numpy as np
import json
from getpass import getpass
from datetime import datetime, timezone
from google.colab import files

# ==================================================
# 1. ENTER API KEY SECURELY
# ==================================================
API_KEY = getpass("Paste your OpenRouteService API key: ")

# ==================================================
# 2. API ENDPOINT (UPDATED)
# ==================================================
url = "https://api.heigit.org/openrouteservice/v2/directions/driving-car/json"

# Coordinates use [longitude, latitude]
locations = {
    "Dubai": [55.2708, 25.2048],
    "Abu Dhabi": [54.3773, 24.4539]
}

# ==================================================
# 3. REQUEST REAL ROUTE DATA
# ==================================================
payload = {
    "coordinates": [
        locations["Dubai"],
        locations["Abu Dhabi"]
    ]
}

headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

print("Requesting route data from OpenRouteService...")

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=30
)

if response.status_code != 200:
    print("API request failed.")
    print("HTTP status:", response.status_code)
    print("Error details:", response.text[:1000])
    response.raise_for_status()

data = response.json()

if not data.get("routes"):
    raise ValueError("The API response contains no routes.")

summary = data["routes"][0]["summary"]

# ==================================================
# 4. CREATE A DATAFRAME FROM THE API RESPONSE
# ==================================================
raw_df = pd.DataFrame([{
    "origin": "Dubai",
    "destination": "Abu Dhabi",
    "distance_m": summary.get("distance"),
    "duration_s": summary.get("duration"),
    "data_source": "OpenRouteService API",
    "data_type": "Real route estimate",
    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}])

print("\nRaw API data:")
display(raw_df)

# ==================================================
# 5. CLEAN DATA WITH PANDAS AND NUMPY
# ==================================================
df = raw_df.copy()

# Convert numeric columns safely
for column in ["distance_m", "duration_s"]:
    df[column] = pd.to_numeric(
        df[column], errors="coerce"
    )

# Validate missing, zero and negative measurements
valid_measurements = (
    df["distance_m"].notna()
    & df["duration_s"].notna()
    & (df["distance_m"] > 0)
    & (df["duration_s"] > 0)
)

df["data_quality_status"] = np.where(
    valid_measurements,
    "Valid",
    "Check required"
)

# Convert metres to kilometres
df["distance_km"] = (
    df["distance_m"] / 1000
).round(2)

# Convert seconds to minutes
df["duration_min"] = (
    df["duration_s"] / 60
).round(2)

# Calculate estimated average speed safely
df["estimated_speed_kmh"] = np.where(
    valid_measurements,
    df["distance_m"] / df["duration_s"] * 3.6,
    np.nan
)

df["estimated_speed_kmh"] = (
    df["estimated_speed_kmh"].round(2)
)

# Flag missing values rather than inventing replacements
df["missing_measurement_count"] = (
    df[["distance_m", "duration_s"]]
    .isna()
    .sum(axis=1)
)

# Remove duplicate origin-destination records
df = df.drop_duplicates(
    subset=["origin", "destination"],
    keep="last"
).reset_index(drop=True)

# ==================================================
# 6. SELECT CLEANED, POWER BI-READY COLUMNS
# ==================================================
clean_df = df[[
    "origin",
    "destination",
    "distance_km",
    "duration_min",
    "estimated_speed_kmh",
    "data_source",
    "data_type",
    "retrieved_at_utc",
    "data_quality_status",
    "missing_measurement_count"
]].copy()

print("\nCleaned route data:")
display(clean_df)

print("\nData quality summary:")
print(clean_df["data_quality_status"].value_counts())

# ==================================================
# 7. EXPORT CLEANED DATA TO JSON
# ==================================================
output_file = "uae_routes_cleaned.json"

# Convert missing numerical values to JSON null
records = clean_df.astype(object).where(
    pd.notna(clean_df), None
).to_dict(orient="records")

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(
        records,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

print(f"\nCleaned JSON saved as: {output_file}")

# ==================================================
# 8. DOWNLOAD THE JSON FILE
# ==================================================
files.download(output_file)


Paste your OpenRouteService API key: ··········
Requesting route data from OpenRouteService...

Raw API data:


,origin,destination,distance_m,duration_s,data_source,data_type,retrieved_at_utc
0,Dubai,Abu Dhabi,139127.4,4762.4,OpenRouteService API,Real route estimate,2026-10-10T14:53:36.851129+00:00



Cleaned route data:


,origin,destination,distance_km,duration_min,estimated_speed_kmh,data_source,data_type,retrieved_at_utc,data_quality_status,missing_measurement_count
0,Dubai,Abu Dhabi,139.13,79.37,105.17,OpenRouteService API,Real route estimate,2026-10-10T14:53:36.851129+00:00,Valid,0



Data quality summary:
data_quality_status
Valid    1
Name: count, dtype: int64

Cleaned JSON saved as: uae_routes_cleaned.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import requests
import pandas as pd
import numpy as np
import json
import time
from getpass import getpass
from datetime import datetime, timezone
from itertools import permutations
from google.colab import files

# =====================================================
# 1. SECURE API KEY
# =====================================================
API_KEY = getpass("Enter your OpenRouteService API key: ")

# =====================================================
# 2. UAE CITY COORDINATES
# Format: longitude, latitude
# =====================================================
cities = {
    "Dubai": [55.2708, 25.2048],
    "Abu Dhabi": [54.3773, 24.4539],
    "Sharjah": [55.4033, 25.3463],
    "Ajman": [55.5136, 25.4052],
    "Ras Al Khaimah": [55.9432, 25.7895],
    "Fujairah": [56.3265, 25.1288],
    "Al Ain": [55.7447, 24.1917]
}

API_URL = (
    "https://api.heigit.org/openrouteservice/"
    "v2/directions/driving-car/json"
)

# =====================================================
# 3. CREATE ALL DIRECTED ROUTE PAIRS
# Seven cities = 7 x 6 = 42 possible directed routes
# =====================================================
route_pairs = list(permutations(cities.keys(), 2))

print("Cities:", len(cities))
print("Routes to request:", len(route_pairs))

# =====================================================
# 4. REQUEST ROUTES FROM THE API
# =====================================================
session = requests.Session()
session.headers.update({
    "Authorization": API_KEY,
    "Content-Type": "application/json"
})

records = []
retrieved_at = datetime.now(timezone.utc).isoformat()

for number, (origin, destination) in enumerate(
    route_pairs, start=1
):
    print(
        f"{number}/{len(route_pairs)}: "
        f"{origin} -> {destination}"
    )

    payload = {
        "coordinates": [
            cities[origin],
            cities[destination]
        ]
    }

    route_record = {
        "route_id": f"R{number:03d}",
        "origin": origin,
        "destination": destination,
        "distance_m": None,
        "duration_s": None,
        "data_source": "OpenRouteService API",
        "data_type": "Real route estimate",
        "retrieved_at_utc": retrieved_at,
        "api_status": "Failed"
    }

    for attempt in range(3):
        try:
            response = session.post(
                API_URL,
                json=payload,
                timeout=30
            )

            if response.status_code == 200:
                result = response.json()

                if result.get("routes"):
                    summary = result["routes"][0]["summary"]
                    route_record["distance_m"] = summary.get(
                        "distance"
                    )
                    route_record["duration_s"] = summary.get(
                        "duration"
                    )
                    route_record["api_status"] = "Success"
                else:
                    route_record["api_status"] = "No route returned"

                break

            # Retry temporary server or rate-limit errors
            if response.status_code == 429 or response.status_code >= 500:
                if attempt < 2:
                    wait_seconds = 5 * (attempt + 1)
                    print(
                        f"Temporary API error "
                        f"{response.status_code}; "
                        f"retrying in {wait_seconds}s"
                    )
                    time.sleep(wait_seconds)
                    continue

            route_record["api_status"] = (
                f"HTTP {response.status_code}"
            )
            print("API error:", response.text[:250])
            break

        except requests.RequestException as error:
            route_record["api_status"] = "Request error"
            if attempt < 2:
                time.sleep(5 * (attempt + 1))
            else:
                print("Request failed:", str(error)[:250])

    records.append(route_record)

    # Avoid sending requests too quickly
    time.sleep(1)

# =====================================================
# 5. CREATE DATAFRAME WITH PANDAS
# =====================================================
df = pd.DataFrame(records)

for column in ["distance_m", "duration_s"]:
    df[column] = pd.to_numeric(
        df[column], errors="coerce"
    )

# =====================================================
# 6. CLEAN AND VALIDATE WITH PANDAS + NUMPY
# =====================================================
valid = (
    df["api_status"].eq("Success")
    & df["distance_m"].notna()
    & df["duration_s"].notna()
    & df["distance_m"].gt(0)
    & df["duration_s"].gt(0)
)

df["data_quality_status"] = np.where(
    valid, "Valid", "Check required"
)

df["distance_km"] = (
    df["distance_m"] / 1000
).round(2)

df["duration_min"] = (
    df["duration_s"] / 60
).round(2)

df["estimated_speed_kmh"] = np.where(
    valid,
    df["distance_m"] / df["duration_s"] * 3.6,
    np.nan
)

df["estimated_speed_kmh"] = (
    df["estimated_speed_kmh"].round(2)
)

# Flag duplicate origin-destination pairs
df["duplicate_route"] = df.duplicated(
    subset=["origin", "destination"],
    keep="first"
)

# =====================================================
# 7. SELECT FINAL ANALYSIS-READY COLUMNS
# =====================================================
clean_df = df[[
    "route_id",
    "origin",
    "destination",
    "distance_km",
    "duration_min",
    "estimated_speed_kmh",
    "data_source",
    "data_type",
    "retrieved_at_utc",
    "api_status",
    "data_quality_status",
    "duplicate_route"
]].copy()

# =====================================================
# 8. DISPLAY RESULTS AND DATA QUALITY
# =====================================================
print("\nROUTE DATA")
display(clean_df)

print("\nDATA QUALITY SUMMARY")
print(clean_df["data_quality_status"].value_counts())

print("\nAPI STATUS SUMMARY")
print(clean_df["api_status"].value_counts())

print(
    "\nSuccessful routes:",
    int(df["api_status"].eq("Success").sum()),
    "out of",
    len(df)
)

# =====================================================
# 9. EXPORT CLEANED DATA AS JSON
# =====================================================
output_file = "uae_routes_cleaned.json"

records_json = clean_df.astype(object).where(
    pd.notna(clean_df), None
).to_dict(orient="records")

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(
        records_json,
        file,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

print("\nSaved:", output_file)

# =====================================================
# 10. DOWNLOAD JSON TO YOUR LAPTOP
# =====================================================
files.download(output_file)

Enter your OpenRouteService API key: ··········
Cities: 7
Routes to request: 42
1/42: Dubai -> Abu Dhabi
2/42: Dubai -> Sharjah
3/42: Dubai -> Ajman
4/42: Dubai -> Ras Al Khaimah
5/42: Dubai -> Fujairah
6/42: Dubai -> Al Ain
7/42: Abu Dhabi -> Dubai
8/42: Abu Dhabi -> Sharjah
9/42: Abu Dhabi -> Ajman
10/42: Abu Dhabi -> Ras Al Khaimah
11/42: Abu Dhabi -> Fujairah
12/42: Abu Dhabi -> Al Ain
13/42: Sharjah -> Dubai
14/42: Sharjah -> Abu Dhabi
15/42: Sharjah -> Ajman
16/42: Sharjah -> Ras Al Khaimah
17/42: Sharjah -> Fujairah
18/42: Sharjah -> Al Ain
19/42: Ajman -> Dubai
20/42: Ajman -> Abu Dhabi
21/42: Ajman -> Sharjah
22/42: Ajman -> Ras Al Khaimah
23/42: Ajman -> Fujairah
24/42: Ajman -> Al Ain
25/42: Ras Al Khaimah -> Dubai
26/42: Ras Al Khaimah -> Abu Dhabi
27/42: Ras Al Khaimah -> Sharjah
28/42: Ras Al Khaimah -> Ajman
29/42: Ras Al Khaimah -> Fujairah
30/42: Ras Al Khaimah -> Al Ain
31/42: Fujairah -> Dubai
32/42: Fujairah -> Abu Dhabi
33/42: Fujairah -> Sharjah
34/42: Fujairah ->

,route_id,origin,destination,distance_km,duration_min,estimated_speed_kmh,data_source,data_type,retrieved_at_utc,api_status,data_quality_status,duplicate_route
0,R001,Dubai,Abu Dhabi,139.13,79.37,105.17,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
1,R002,Dubai,Sharjah,25.73,26.05,59.25,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
2,R003,Dubai,Ajman,43.97,35.64,74.04,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
3,R004,Dubai,Ras Al Khaimah,107.21,67.46,95.35,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
4,R005,Dubai,Fujairah,124.09,81.86,90.95,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
5,R006,Dubai,Al Ain,137.99,91.17,90.81,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
6,R007,Abu Dhabi,Dubai,139.75,79.77,105.12,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
7,R008,Abu Dhabi,Sharjah,162.56,99.41,98.11,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
8,R009,Abu Dhabi,Ajman,181.05,110.48,98.33,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False
9,R010,Abu Dhabi,Ras Al Khaimah,244.29,142.30,103.00,OpenRouteService API,Real route estimate,2026-10-10T15:05:04.227776+00:00,Success,Valid,False



DATA QUALITY SUMMARY
data_quality_status
Valid    42
Name: count, dtype: int64

API STATUS SUMMARY
api_status
Success    42
Name: count, dtype: int64

Successful routes: 42 out of 42

Saved: uae_routes_cleaned.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>